## Create SQL Functions for your Genie Agents

### Set the Execution Context

In [ ]:
%sql
USE SCHEMA genie_lab;

### Create the VIP Customer Classification Function

In [ ]:
%sql
CREATE OR REPLACE FUNCTION get_vip_customers()
RETURNS TABLE (
    customer_name STRING,
    region STRING,
    lifetime_revenue DOUBLE
)
COMMENT 'Returns VIP customers with lifetime net revenue of at least 15,000 dollars'
RETURN

SELECT
    c.customer_name,
    c.region,
    SUM(o.gross_revenue - o.discount_amount) AS lifetime_revenue

FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id

WHERE o.order_status = 'COMPLETED'

GROUP BY
    c.customer_name,
    c.region

HAVING SUM(o.gross_revenue - o.discount_amount) >= 15000;

In [ ]:
%sql
SELECT *
FROM get_vip_customers();

### Create the Performance Function

In [ ]:
%sql
CREATE OR REPLACE FUNCTION get_revenue_performance()
RETURNS TABLE (
    region STRING,
    net_revenue DOUBLE,
    performance STRING
)
COMMENT 'Returns net revenue by region and classifies regional revenue performance.'
RETURN

SELECT
    c.region,

    SUM(o.gross_revenue - o.discount_amount) AS net_revenue,

    CASE
        WHEN SUM(o.gross_revenue - o.discount_amount) >= 20000
            THEN 'Excellent'
        WHEN SUM(o.gross_revenue - o.discount_amount) >= 10000
            THEN 'Strong'
        WHEN SUM(o.gross_revenue - o.discount_amount) >= 5000
            THEN 'Moderate'
        ELSE 'Low'
    END AS performance

FROM orders o

JOIN customers c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'COMPLETED'

GROUP BY c.region;

In [ ]:
%sql
SELECT *
FROM get_revenue_performance();